In [13]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)
from utils.user_utils import get_clf_eval, get_model_train_eval

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

In [15]:
train['var3'].replace(-999999, 2, inplace=True)
test['var3'].replace(-999999, 2, inplace=True)

y = train['TARGET']
X = train.drop(['ID', 'TARGET'], axis=1)

X_test_only = test.drop(['ID'], axis=1)

In [16]:
# df.info()

# print("\n 결측값의 수:", df.isna().sum().sum())

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 76020 entries, 0 to 76019
# Columns: 369 entries, var3 to var38
# dtypes: float64(111), int64(258)
# memory usage: 214.0 MB

#  결측값의 수: 0

In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Train shape: (60816, 369)
Validation shape: (15204, 369)


In [18]:
# -----------------------------
# 4. StandardScaler
# -----------------------------
scaler = StandardScaler()
scaler.fit(X_train)          # ✔ train만 fit

X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_only)   # 여기서 test도 transform만!

In [19]:
# 레이블의 분포 확인
cust_cnt = y.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [52]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        self.base_model.fit(X, y)
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


In [57]:
# -----------------------------
# 5. 모델 학습: XGBoost
# -----------------------------

xgb = XGBClassifier(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=3,
    scale_pos_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train_scaled, y_train)
pred_val = xgb.predict(X_val_scaled)
proba_val = xgb.predict_proba(X_val_scaled)[:,1]

get_model_train_eval(xgb,'XGB_100_est700_lr0.03_max3_spw3',
    X_train, X_val,
    y_train, y_val
)

best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
# pred_best = (proba_val >= best_threshold).astype(int)
# get_clf_eval(y_val, pred_best, proba_val)
threshold_model = ThresholdModel(xgb, best_threshold)

get_model_train_eval(threshold_model,'XGB_100_est700_lr0.03_max3_spw3_thr',
    X_train, X_val,
    y_train, y_val
)

✓ 모델 저장 완료: models\XGB_100_est700_lr0.03_max3_spw3.pkl
  파일 크기: 0.77 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8501, 정확도: 0.9537, 정밀도: 0.2548, 재현율: 0.0880, F1: 0.1309
오차행렬:
[[14447   155]
 [  549    53]]
실행 시간: 5.995568513870239

Best Threshold: 0.32, Best F1: 0.2889
✓ 모델 저장 완료: models\XGB_100_est700_lr0.03_max3_spw3_thr.pkl
  파일 크기: 0.77 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8501, 정확도: 0.9064, 정밀도: 0.2066, 재현율: 0.4801, F1: 0.2889
오차행렬:
[[13492  1110]
 [  313   289]]
실행 시간: 6.012495756149292


In [58]:
class ThresholdModel_rf:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # fit을 무시하고, 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [ ]:
# -----------------------------
# 6. RandomForest
# -----------------------------

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight={0:1, 1:2},
    random_state=0,
    n_jobs=-1
)

rf.fit(X_train, y_train)    # 랜포는 스케일링 필요 없음
rf_pred = rf.predict(X_val)
rf_proba = rf.predict_proba(X_val)[:, 1]

get_model_train_eval(rf,'RF_100_est300_max20_class1vs2',
    X_train, X_val,
    y_train, y_val
)

# 🔥 반드시 초기화해야 함
best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (rf_proba >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
# pred_best = (rf_proba >= best_threshold).astype(int)
# get_clf_eval(y_val, pred_best, rf_proba)
threshold_model_rf = ThresholdModel_rf(rf, best_threshold)

get_model_train_eval(threshold_model_rf,'RF_100_est300_max20_class1vs2_thr',
    X_train, X_val,
    y_train, y_val
)

✓ 모델 저장 완료: models\RF_100_est300_max20_class1vs2.pkl
  파일 크기: 49.76 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8356, 정확도: 0.9603, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
오차행렬:
[[14600     2]
 [  602     0]]
실행 시간: 6.614229917526245

Best Threshold: 0.21, Best F1: 0.2800
✓ 모델 저장 완료: models\RF_100_est300_max20_class1vs2_thr.pkl
  파일 크기: 49.76 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8356, 정확도: 0.9083, 정밀도: 0.2031, 재현율: 0.4502, F1: 0.2800
오차행렬:
[[13539  1063]
 [  331   271]]
실행 시간: 0.35898756980895996


In [23]:
# ============================================
# 6. TEST.CSV에 대한 최종 예측
#
# TARGET=1(불만족)을 얼마나 잘 잡아내는지
# 즉, “잠재적으로 문제가 생길 고객”을 잘 찾아내는지?
#
# 불만족(=1) 고객 비율이 너무 낮기 때문에
# F1 score과 recall이 낮게 나오는 것이 정상이고,
# AUC를 중심으로 보는 게 맞아.
# ============================================

# XGBoost test 예측값
xgb_test_pred = xgb.predict_proba(X_test_scaled)[:, 1]

# RandomForest test 예측값
rf_test_pred = rf.predict_proba(X_test_only)[:, 1]

print("\n===== FINAL TEST PREDICTIONS =====")
print("\nXGBoost Test Predictions (probability of TARGET=1(불만족)):")
print(xgb_test_pred[:20])   # 상위 20개만 미리보기

print("\nRandomForest Test Predictions (probability of TARGET=1(불만족)):")
print(rf_test_pred[:20])    # 상위 20개만 미리보기


===== FINAL TEST PREDICTIONS =====

XGBoost Test Predictions (probability of TARGET=1(불만족)):
[0.01531759 0.01531759 0.01483977 0.01531759 0.01531759 0.01531759
 0.02546733 0.01531759 0.01439818 0.01546898 0.01531759 0.01531759
 0.01786462 0.01531759 0.01054986 0.01547328 0.01483977 0.01531759
 0.01531759 0.01776981]

RandomForest Test Predictions (probability of TARGET=1(불만족)):
[0.06008547 0.06054385 0.0215892  0.07928155 0.02017998 0.24645532
 0.06404603 0.23378981 0.03953955 0.04020289 0.04272866 0.02776394
 0.03454718 0.02182546 0.03519108 0.05251217 0.12909037 0.02113416
 0.02213186 0.03631033]


In [24]:
best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"Best Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
pred_best = (proba_val >= best_threshold).astype(int)
get_clf_eval(y_val, pred_best, proba_val)

# -----------------------------
# RandomForest Threshold Optimization for F1
# -----------------------------



print(f"[RF] Best Threshold: {best_threshold:.2f}, Best F1 Score: {best_f1:.4f}")

# 최적 threshold로 평가 지표 출력
rf_pred_best = (rf_proba >= best_threshold).astype(int)
get_clf_eval(y_val, rf_pred_best, rf_proba, model_name='RF_best')


Best Threshold: 0.13, Best F1: 0.2974
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8509, 정확도: 0.9036, 정밀도: 0.2090, 재현율: 0.5150, F1: 0.2974
오차행렬:
[[13429  1173]
 [  292   310]]
[RF] Best Threshold: 0.13, Best F1 Score: 0.2974
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8264, 정확도: 0.8630, 정밀도: 0.1675, 재현율: 0.6196, F1: 0.2637
오차행렬:
[[12748  1854]
 [  229   373]]
